In [14]:
#N.1
#Extracted raw binary files

import struct
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from itertools import cycle
import time

BASE = '/kaggle/input/datasets/mtalhasohail/emnist-balanced'   

# read the binary image file
with open(f'{BASE}/emnist-balanced-train-images-idx3-ubyte', 'rb') as f:
    magic, n, rows, cols = struct.unpack('>IIII', f.read(16))
    x_train_all = np.frombuffer(f.read(), dtype=np.uint8).reshape(n, rows*cols)

with open(f'{BASE}/emnist-balanced-test-images-idx3-ubyte', 'rb') as f:
    magic, n, rows, cols = struct.unpack('>IIII', f.read(16))
    x_test_all = np.frombuffer(f.read(), dtype=np.uint8).reshape(n, rows*cols)

# read the binary label file
with open(f'{BASE}/emnist-balanced-train-labels-idx1-ubyte', 'rb') as f:
    magic, n = struct.unpack('>II', f.read(8))
    y_train_all = np.frombuffer(f.read(), dtype=np.uint8)

with open(f'{BASE}/emnist-balanced-test-labels-idx1-ubyte', 'rb') as f:
    magic, n = struct.unpack('>II', f.read(8))
    y_test_all = np.frombuffer(f.read(), dtype=np.uint8)

print(f"train: {x_train_all.shape}   test: {x_test_all.shape}")
print(f"labels: {y_train_all.min()} to {y_train_all.max()}")


train: (112800, 784)   test: (18800, 784)
labels: 0 to 46


In [15]:
#N.2
#fixed EMNIST rotation
#Images come rotated 90° and mirrored in the dataset

print("*** original(eminist dataset) ***")
for row in x_train_all[2].reshape(28, 28):
    print(''.join('#' if p > 100 else ' ' for p in row))
print("label:", y_train_all[2])

for row in x_train_all[5].reshape(28, 28):
    print(''.join('#' if p > 100 else ' ' for p in row))
print("label:", y_train_all[5])

# fix train
imgs = x_train_all.reshape(-1, 28, 28)
imgs = np.rot90(imgs, k=3, axes=(1, 2))
imgs = np.flip(imgs, axis=2)
x_train_all = imgs.reshape(-1, 784).copy()

# fix test
imgs = x_test_all.reshape(-1, 28, 28)
imgs = np.rot90(imgs, k=3, axes=(1, 2))
imgs = np.flip(imgs, axis=2)
x_test_all = imgs.reshape(-1, 784).copy()


print("*** pre-processed(eminist dataset) ***")
for row in x_train_all[2].reshape(28, 28):
    print(''.join('#' if p > 100 else ' ' for p in row))
print("label:", y_train_all[2])

for row in x_train_all[5].reshape(28, 28):
    print(''.join('#' if p > 100 else ' ' for p in row))
print("label:", y_train_all[5])


*** original(eminist dataset) ***
                            
                            
                ## # ###    
       ##################   
    #####################   
   ####################     
   ####      ####           
    ###    ####             
          ####              
         ###                
        ###                 
        ###                 
       ###                  
       ###                  
       ###                  
       ###                  
        #####               
         #######            
          ########          
              ######        
                 ####       
                   ###      
                   ###      
                   ##       
                  ##        
                            
                            
                            
label: 43
                            
                            
                            
                            
                            

In [16]:
#C.3
# 6 problems

SCRIPT_NAMES = ["P1_A-G", "P2_H-N", "P3_O-U", "P4_V-b", "P5_d-q", "P6_1-7"]

# (looked up from the emnist-balanced-mapping file)
GROUP_LABELS = [
    [10, 11, 12, 13, 14, 15, 16],   # A B C D E F G
    [17, 18, 19, 20, 21, 22, 23],   # H I J K L M N
    [24, 25, 26, 27, 28, 29, 30],   # O P Q R S T U
    [31, 32, 33, 34, 35, 36, 37],   # V W X Y Z a b
    [38, 39, 40, 41, 42, 43, 44],   # d e f g h n q
    [1,  2,  3,  4,  5,  6,  7],    # 1 2 3 4 5 6 7
]

#5 classes not used: zero, 8, 9, r(45), t(46)
print("6 problems defined, 7 classes each, 42 classes total")



6 problems defined, 7 classes each, 42 classes total


In [17]:
#N.4 
#build 6 datasets
# 7 classes for each problem
#remapping of  labels
#and standardize pixels to mean 0, std 1.

train_datasets = []
test_datasets  = []

for m in range(6):
    labels = GROUP_LABELS[m]

    # --- training data ---
    mask = np.isin(y_train_all, labels)
    x_tr = x_train_all[mask].copy()
    y_tr_raw = y_train_all[mask].copy()

    # remap:[17,18,19,20,21,22,23] -> [0,1,2,3,4,5,6]
    y_tr = np.zeros(len(y_tr_raw), dtype=np.int64)
    for new_label, old_label in enumerate(labels):
        y_tr[y_tr_raw == old_label] = new_label# find pos where y_tr_raw equals old_label & put new label 

    mask = np.isin(y_test_all, labels)
    x_te = x_test_all[mask].copy()
    y_te_raw = y_test_all[mask].copy()

    y_te = np.zeros(len(y_te_raw), dtype=np.int64)
    for new_label, old_label in enumerate(labels):
        y_te[y_te_raw == old_label] = new_label

    x_tr = torch.tensor(x_tr, dtype=torch.float32)
    x_te = torch.tensor(x_te, dtype=torch.float32)
    y_tr = torch.tensor(y_tr, dtype=torch.long)
    y_te = torch.tensor(y_te, dtype=torch.long)

    mean = x_tr.mean()
    std  = x_tr.std()
    x_tr = (x_tr - mean) / std
    x_te = (x_te - mean) / std

    train_datasets.append(TensorDataset(x_tr, y_tr))
    test_datasets.append(TensorDataset(x_te, y_te))

    print(f"{SCRIPT_NAMES[m]}: train {len(x_tr):5d}  test {len(x_te):4d}  "
          f"labels {y_tr.min().item()}-{y_tr.max().item()}  "
          f"mean {x_tr.mean():.2f}  std {x_tr.std():.2f}")



P1_A-G: train 16800  test 2800  labels 0-6  mean -0.00  std 1.00
P2_H-N: train 16800  test 2800  labels 0-6  mean -0.00  std 1.00
P3_O-U: train 16800  test 2800  labels 0-6  mean 0.00  std 1.00
P4_V-b: train 16800  test 2800  labels 0-6  mean 0.00  std 1.00
P5_d-q: train 16800  test 2800  labels 0-6  mean 0.00  std 1.00
P6_1-7: train 16800  test 2800  labels 0-6  mean -0.00  std 1.00


In [18]:
#N.5
#Data loaders & intial settings
BATCH_SIZE = 32
train_loaders = [DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True) for ds in train_datasets]
test_loaders  = [DataLoader(ds, batch_size=256, shuffle=False)       for ds in test_datasets]

DEVICE          = 'cuda' if torch.cuda.is_available() else 'cpu'
INPUT_SIZE      = 784
HIDDEN_SIZE     = 128
NUM_G_SETS      = 4
NUM_PROBLEMS    = 6
CLASSES_PER     = 7
TOTAL_CLASSES   = NUM_PROBLEMS * CLASSES_PER   # 42

LEARNING_RATE   = 0.01
MOMENTUM        = 0.9
NUM_EPOCHS      = 50
STEPS_PER_EPOCH = 10
N_RUNS          = 100   

B_CONFIGS = torch.tensor([
    [1,1,0,0],
    [1,0,1,0],
    [1,0,0,1],
    [0,1,1,0],
    [0,1,0,1],
    [0,0,1,1],
], dtype=torch.float32).to(DEVICE)

print(f"Device: {DEVICE}")
print(f"Runs: {N_RUNS}  Epochs: {NUM_EPOCHS}  Steps/epoch: {STEPS_PER_EPOCH}")



Device: cuda
Runs: 100  Epochs: 50  Steps/epoch: 10


In [19]:
#N.6
#MODEL-1:COMPOUND SYNAPSE
#4 G sets for hidden layer, 4 G sets for output layer.
#Same b selector picks 2-of-4 in both layers.
#No separate output heads the output is also compound.
#Parameters: 4 * [784*128 + 128*7] = 404,992

accs_compound = []

print("Training COMPOUND SYNAPSE...")
t0 = time.time()

for run in range(N_RUNS):

    # 4 hidden G sets: each is [128, 784]
    Ghid_comp = nn.Parameter(torch.empty(4, HIDDEN_SIZE, INPUT_SIZE, device=DEVICE))
    for n in range(4):
        #ran loop because if we intialize with (4,128,784),it will consider convolution
        #4 input channels,128 output channels,784 kernel size
        #fan_in:sqrt(2/input size) will be computed wrong
        #fan_out:sqrt(2/output size)
        #& it will compute wrong fan_in
        nn.init.kaiming_normal_(Ghid_comp.data[n], mode="fan_in", nonlinearity="relu")

    # 4 output G sets: each is [7, 128]
    Gout_comp = nn.Parameter(torch.empty(4, CLASSES_PER, HIDDEN_SIZE, device=DEVICE))
    for n in range(4):
        nn.init.kaiming_normal_(Gout_comp.data[n], mode="fan_in", nonlinearity="relu")

    optimizer_comp = torch.optim.SGD([Ghid_comp, Gout_comp],
                                     lr=LEARNING_RATE, momentum=MOMENTUM)
   
    inf_iters_comp = [cycle(loader) for loader in train_loaders]

    # train
    for epoch in range(NUM_EPOCHS):
        for step in range(STEPS_PER_EPOCH):
            optimizer_comp.zero_grad()
            total_loss = torch.zeros((), device=DEVICE)

            for m in range(6):
                x, y = next(inf_iters_comp[m])
                x, y = x.to(DEVICE), y.to(DEVICE)

                b = B_CONFIGS[m]
                W    = torch.einsum('n,nhi->hi', b, Ghid_comp)   # [128, 784]
                Wout = torch.einsum('n,noh->oh', b, Gout_comp)   # [7, 128]
                h    = F.relu(x @ W.T)                           # [batch, 128]
                out  = h @ Wout.T                                 # [batch, 7]

                total_loss = total_loss + F.cross_entropy(out, y)

            total_loss.backward()
            optimizer_comp.step()

    # evaluate
    correct_all = 0
    total_all   = 0
    with torch.no_grad():
        for m in range(6):
            for x, y in test_loaders[m]:
                x, y = x.to(DEVICE), y.to(DEVICE)
                b = B_CONFIGS[m]
                W    = torch.einsum('n,nhi->hi', b, Ghid_comp)
                Wout = torch.einsum('n,noh->oh', b, Gout_comp)
                h    = F.relu(x @ W.T)
                out  = h @ Wout.T
                correct_all += (out.argmax(1) == y).sum().item()
                total_all   += y.size(0)

    acc = correct_all / total_all * 100
    accs_compound.append(acc)

    if (run + 1) % 10 == 0:
        elapsed = time.time() - t0
        print(f"  run {run+1}/{N_RUNS}  acc {acc:.2f}%  "
              f"running mean {np.mean(accs_compound):.2f}%  "
              f"({elapsed:.0f}s elapsed)")

print(f"\nCOMPOUND done:  {np.mean(accs_compound):.2f}% ± {np.std(accs_compound):.2f}%\n")


Training COMPOUND SYNAPSE...
  run 10/100  acc 91.21%  running mean 91.52%  (39s elapsed)
  run 20/100  acc 92.05%  running mean 91.67%  (77s elapsed)
  run 30/100  acc 91.25%  running mean 91.61%  (116s elapsed)
  run 40/100  acc 91.90%  running mean 91.64%  (154s elapsed)
  run 50/100  acc 91.79%  running mean 91.62%  (193s elapsed)
  run 60/100  acc 91.81%  running mean 91.61%  (231s elapsed)
  run 70/100  acc 91.27%  running mean 91.63%  (270s elapsed)
  run 80/100  acc 91.70%  running mean 91.61%  (310s elapsed)
  run 90/100  acc 92.26%  running mean 91.62%  (349s elapsed)
  run 100/100  acc 91.55%  running mean 91.61%  (387s elapsed)

COMPOUND done:  91.61% ± 0.31%



In [20]:
#N.7
#MODEL 2: b=[1,0,0,0]
#only G1 used
#output layer: 6*7=42 neurons
#parameters: 784*128 + 128*42 = 105,728

accs_b1000 = []

print("Training b=[1,0,0,0]...")
t0 = time.time()

for run in range(N_RUNS):

    # hidden: 4 G sets, but only G[0] will ever be selected
    Ghid_b1000 = nn.Parameter(torch.empty(4, HIDDEN_SIZE, INPUT_SIZE, device=DEVICE))
    for n in range(4):
        nn.init.kaiming_normal_(Ghid_b1000.data[n], mode="fan_in", nonlinearity="relu")

    # output: one shared layer with 42 neurons (not compound)
    Wout_b1000 = nn.Parameter(torch.empty(TOTAL_CLASSES, HIDDEN_SIZE, device=DEVICE))
    nn.init.kaiming_normal_(Wout_b1000.data, mode="fan_in", nonlinearity="relu")

    b_fixed = torch.tensor([1, 0, 0, 0], dtype=torch.float32, device=DEVICE)

    optimizer_b1000 = torch.optim.SGD([Ghid_b1000, Wout_b1000],
                                      lr=LEARNING_RATE, momentum=MOMENTUM)
    inf_iters_b1000 = [cycle(loader) for loader in train_loaders]

    # train
    for epoch in range(NUM_EPOCHS):
        for step in range(STEPS_PER_EPOCH):
            optimizer_b1000.zero_grad()
            total_loss = torch.zeros((), device=DEVICE)

            for m in range(6):
                x, y = next(inf_iters_b1000[m])
                x, y = x.to(DEVICE), y.to(DEVICE)

                W = torch.einsum('n,nhi->hi', b_fixed, Ghid_b1000)   # always G[0]
                h = F.relu(x @ W.T)                                   # [batch, 128]
                out = h @ Wout_b1000.T                                 # [batch, 42]


                #all 42 boxes:
                #0   1   2   3   4   5   6  | 7   8   9   10  11  12  13 | 14 ...
                #0.1 0.2 0.1 0.3 0.1 0.1 0.1| 0.1 0.2 0.1 0.8 0.2 0.1 0.1| 0.1 ...

                y_shifted = y + m * CLASSES_PER
                total_loss = total_loss + F.cross_entropy(out, y_shifted)

            total_loss.backward()
            optimizer_b1000.step()

    # evaluate — only look at this problem's 7 columns of the 42
    correct_all = 0
    total_all   = 0
    with torch.no_grad():
        for m in range(6):
            for x, y in test_loaders[m]:
                x, y = x.to(DEVICE), y.to(DEVICE)
                W = torch.einsum('n,nhi->hi', b_fixed, Ghid_b1000)
                h = F.relu(x @ W.T)
                out = h @ Wout_b1000.T                              # [batch, 42]
                #slice out boxes 7-13 only:
                #7   8   9   10  11  12  13
                #0.1 0.2 0.1 0.8 0.2 0.1 0.1
                out_slice = out[:, m*CLASSES_PER : (m+1)*CLASSES_PER]  # [batch, 7]
                correct_all += (out_slice.argmax(1) == y).sum().item()
                total_all   += y.size(0)

    acc = correct_all / total_all * 100
    accs_b1000.append(acc)

    if (run + 1) % 10 == 0:
        elapsed = time.time() - t0
        print(f"  run {run+1}/{N_RUNS}  acc {acc:.2f}%  "
              f"running mean {np.mean(accs_b1000):.2f}%  "
              f"({elapsed:.0f}s elapsed)")

print(f"\nb=1000 done:  {np.mean(accs_b1000):.2f}% ± {np.std(accs_b1000):.2f}%\n")


Training b=[1,0,0,0]...
  run 10/100  acc 92.29%  running mean 92.47%  (35s elapsed)
  run 20/100  acc 92.09%  running mean 92.39%  (70s elapsed)
  run 30/100  acc 92.71%  running mean 92.41%  (106s elapsed)
  run 40/100  acc 91.96%  running mean 92.37%  (141s elapsed)
  run 50/100  acc 92.38%  running mean 92.39%  (176s elapsed)
  run 60/100  acc 92.73%  running mean 92.40%  (211s elapsed)
  run 70/100  acc 92.38%  running mean 92.39%  (245s elapsed)
  run 80/100  acc 92.67%  running mean 92.39%  (280s elapsed)
  run 90/100  acc 92.33%  running mean 92.39%  (315s elapsed)
  run 100/100  acc 92.28%  running mean 92.37%  (350s elapsed)

b=1000 done:  92.37% ± 0.25%



In [21]:
#N.8
#MODEL 3: SIX INDEPENDENT NETWORKS
#separate weights for hidden layer,output layer for each problem
#parameters: 101,248 each, 607,488 total.

accs_six = []

print("Training 6 INDEPENDENT networks...")
t0 = time.time()

for run in range(N_RUNS):

    # 6 separate hidden weights and 6 separate output weights
    W_six    = []
    Wout_six = []
    opts_six = []

    for m in range(6):
        w_hid = nn.Parameter(torch.empty(HIDDEN_SIZE, INPUT_SIZE, device=DEVICE))
        nn.init.kaiming_normal_(w_hid.data, mode="fan_in", nonlinearity="relu")

        w_out = nn.Parameter(torch.empty(CLASSES_PER, HIDDEN_SIZE, device=DEVICE))
        nn.init.kaiming_normal_(w_out.data, mode="fan_in", nonlinearity="relu")

        W_six.append(w_hid)
        Wout_six.append(w_out)
        opts_six.append(torch.optim.SGD([w_hid, w_out],
                                        lr=LEARNING_RATE, momentum=MOMENTUM))

    inf_iters_six = [cycle(loader) for loader in train_loaders]

    # train each problem independently
    for epoch in range(NUM_EPOCHS):
        for m in range(6):
            for step in range(STEPS_PER_EPOCH):
                opts_six[m].zero_grad()

                x, y = next(inf_iters_six[m])
                x, y = x.to(DEVICE), y.to(DEVICE)

                h   = F.relu(x @ W_six[m].T)          # [batch, 128]
                out = h @ Wout_six[m].T                # [batch, 7]
                loss = F.cross_entropy(out, y)

                loss.backward()
                opts_six[m].step()

    # evaluate
    correct_all = 0
    total_all   = 0
    with torch.no_grad():
        for m in range(6):
            for x, y in test_loaders[m]:
                x, y = x.to(DEVICE), y.to(DEVICE)
                h   = F.relu(x @ W_six[m].T)
                out = h @ Wout_six[m].T
                correct_all += (out.argmax(1) == y).sum().item()
                total_all   += y.size(0)

    acc = correct_all / total_all * 100
    accs_six.append(acc)

    if (run + 1) % 10 == 0:
        elapsed = time.time() - t0
        print(f"  run {run+1}/{N_RUNS}  acc {acc:.2f}%  "
              f"running mean {np.mean(accs_six):.2f}%  "
              f"({elapsed:.0f}s elapsed)")

print(f"\n6 independent done:  {np.mean(accs_six):.2f}% ± {np.std(accs_six):.2f}%\n")



Training 6 INDEPENDENT networks...
  run 10/100  acc 92.28%  running mean 92.38%  (39s elapsed)
  run 20/100  acc 92.77%  running mean 92.38%  (78s elapsed)
  run 30/100  acc 92.38%  running mean 92.40%  (116s elapsed)
  run 40/100  acc 92.43%  running mean 92.40%  (154s elapsed)
  run 50/100  acc 92.52%  running mean 92.42%  (193s elapsed)
  run 60/100  acc 92.11%  running mean 92.42%  (232s elapsed)
  run 70/100  acc 92.58%  running mean 92.42%  (271s elapsed)
  run 80/100  acc 92.10%  running mean 92.42%  (310s elapsed)
  run 90/100  acc 92.23%  running mean 92.42%  (350s elapsed)
  run 100/100  acc 92.65%  running mean 92.42%  (389s elapsed)

6 independent done:  92.42% ± 0.20%



In [22]:
#N.9
#MODEL 4: SINGLE WIDE 490-NEURON NETWORK
#42 output neurons
#same parameters.as compound
# 784*490 + 490*42 = 404,740

WIDE = 490
accs_wide = []

print(f"Training WIDE {WIDE}-NEURON network...")
t0 = time.time()

for run in range(N_RUNS):

    W_wide = nn.Parameter(torch.empty(WIDE, INPUT_SIZE, device=DEVICE))
    nn.init.kaiming_normal_(W_wide.data, mode="fan_in", nonlinearity="relu")

    Wout_wide = nn.Parameter(torch.empty(TOTAL_CLASSES, WIDE, device=DEVICE))
    nn.init.kaiming_normal_(Wout_wide.data, mode="fan_in", nonlinearity="relu")

    optimizer_wide = torch.optim.SGD([W_wide, Wout_wide],
                                     lr=LEARNING_RATE, momentum=MOMENTUM)
    inf_iters_wide = [cycle(loader) for loader in train_loaders]

    # train
    for epoch in range(NUM_EPOCHS):
        for step in range(STEPS_PER_EPOCH):
            optimizer_wide.zero_grad()
            total_loss = torch.zeros((), device=DEVICE)

            for m in range(6):
                x, y = next(inf_iters_wide[m])
                x, y = x.to(DEVICE), y.to(DEVICE)

                h   = F.relu(x @ W_wide.T)            # [batch, 490]
                out = h @ Wout_wide.T                  # [batch, 42]

                y_shifted = y + m * CLASSES_PER
                total_loss = total_loss + F.cross_entropy(out, y_shifted)

            total_loss.backward()
            optimizer_wide.step()

    # evaluate — slice to this problem's 7 columns
    correct_all = 0
    total_all   = 0
    with torch.no_grad():
        for m in range(6):
            for x, y in test_loaders[m]:
                x, y = x.to(DEVICE), y.to(DEVICE)
                h   = F.relu(x @ W_wide.T)
                out = h @ Wout_wide.T
                out_slice = out[:, m*CLASSES_PER : (m+1)*CLASSES_PER]
                correct_all += (out_slice.argmax(1) == y).sum().item()
                total_all   += y.size(0)

    acc = correct_all / total_all * 100
    accs_wide.append(acc)

    if (run + 1) % 10 == 0:
        elapsed = time.time() - t0
        print(f"  run {run+1}/{N_RUNS}  acc {acc:.2f}%  "
              f"running mean {np.mean(accs_wide):.2f}%  "
              f"({elapsed:.0f}s elapsed)")

print(f"\nWide {WIDE} done:  {np.mean(accs_wide):.2f}% ± {np.std(accs_wide):.2f}%\n")



Training WIDE 490-NEURON network...
  run 10/100  acc 93.72%  running mean 93.62%  (30s elapsed)
  run 20/100  acc 93.52%  running mean 93.55%  (61s elapsed)
  run 30/100  acc 93.64%  running mean 93.56%  (91s elapsed)
  run 40/100  acc 93.91%  running mean 93.61%  (120s elapsed)
  run 50/100  acc 93.57%  running mean 93.61%  (150s elapsed)
  run 60/100  acc 93.72%  running mean 93.62%  (180s elapsed)
  run 70/100  acc 93.80%  running mean 93.62%  (210s elapsed)
  run 80/100  acc 93.39%  running mean 93.61%  (240s elapsed)
  run 90/100  acc 93.76%  running mean 93.60%  (270s elapsed)
  run 100/100  acc 93.47%  running mean 93.61%  (300s elapsed)

Wide 490 done:  93.61% ± 0.25%



In [23]:
#N.10 FINAL COMPARISON

print("=" * 60)
print(f"{'Model':<28}{'Hidden params':>14}{'Mean':>10}{'Std':>8}")
print("-" * 60)
print(f"{'Compound (4G, b-selected)':<28}{'404,992':>14}"
      f"{np.mean(accs_compound):>9.2f}%{np.std(accs_compound):>7.2f}%")
print(f"{'b=1000 (only G1)':<28}{'105,728':>14}"
      f"{np.mean(accs_b1000):>9.2f}%{np.std(accs_b1000):>7.2f}%")
print(f"{'6 independent':<28}{'607,488':>14}"
      f"{np.mean(accs_six):>9.2f}%{np.std(accs_six):>7.2f}%")
print(f"{'Wide 490 shared':<28}{'404,740':>14}"
      f"{np.mean(accs_wide):>9.2f}%{np.std(accs_wide):>7.2f}%")
print("=" * 60)
print(f"\n(each trained {N_RUNS} times, {NUM_EPOCHS} epochs, "
      f"{STEPS_PER_EPOCH} steps/epoch, SGD momentum={MOMENTUM}, lr={LEARNING_RATE})")

Model                        Hidden params      Mean     Std
------------------------------------------------------------
Compound (4G, b-selected)          404,992    91.61%   0.31%
b=1000 (only G1)                   105,728    92.37%   0.25%
6 independent                      607,488    92.42%   0.20%
Wide 490 shared                    404,740    93.61%   0.25%

(each trained 100 times, 50 epochs, 10 steps/epoch, SGD momentum=0.9, lr=0.01)
